In [2]:
from pathlib import Path
from datetime import date

In [4]:
import pandas as pd
import yfinance as yf

In [7]:
from pathlib import Path

# Define project folders using the current notebook location
PROJECT_DIR = Path.cwd().parent  # .parent goes up one folder level, adjust as needed
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


In [8]:
# Select stocks and historical period
TICKERS = ["AAPL", "MSFT", "NVDA"]
PERIOD = "5y"

In [9]:
# Download and validate each stock
all_stocks = []

for ticker in TICKERS:
    print(f"\nDownloading {ticker}...")

    try:
        stock = yf.Ticker(ticker)
        df = stock.history(
            period=PERIOD,
            interval="1d",
            auto_adjust=True
        )

        if df.empty:
            print(f"WARNING: No data returned for {ticker}.")
            continue

        # Keep the fields needed for analysis
        df = df[["Open", "High", "Low", "Close", "Volume"]].copy()

        # Make the date index a regular column
        df.index = pd.to_datetime(df.index).tz_localize(None)
        df.index.name = "Date"
        df = df.reset_index()

        # Add the stock ticker
        df["Ticker"] = ticker

        # Basic data quality checks
        df = df.drop_duplicates(subset=["Date"])
        df = df.sort_values("Date")

        price_columns = ["Open", "High", "Low", "Close"]
        missing_prices = df[price_columns].isna().sum().sum()
        invalid_prices = (df[price_columns] <= 0).sum().sum()

        if missing_prices > 0 or invalid_prices > 0:
            print(
                f"WARNING: {ticker} has "
                f"{missing_prices} missing and "
                f"{invalid_prices} non-positive price values."
            )

        # Save individual stock data
        output_file = RAW_DIR / f"{ticker}_historical.csv"
        df.to_csv(output_file, index=False)

        all_stocks.append(df)

        print(f"Rows downloaded: {len(df):,}")
        print(f"Date range: {df['Date'].min().date()} "
              f"to {df['Date'].max().date()}")
        print(f"Saved to: {output_file}")

    except Exception as error:
        print(f"ERROR downloading {ticker}: {error}")


Rows downloaded: 1,254
Date range: 2021-10-22 to 2026-10-21
Saved to: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/data/raw/AAPL_historical.csv

Rows downloaded: 1,254
Date range: 2021-10-22 to 2026-10-21
Saved to: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/data/raw/MSFT_historical.csv

Rows downloaded: 1,254
Date range: 2021-10-22 to 2026-10-21
Saved to: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/data/raw/NVDA_historical.csv


In [10]:
# Combine all successfully downloaded stocks
if not all_stocks:
    raise RuntimeError(
        "No stock data downloaded. Check your internet connection "
        "and try again."
    )

combined = pd.concat(all_stocks, ignore_index=True)
combined = combined.sort_values(["Date", "Ticker"])

combined_file = RAW_DIR / "all_stocks_historical.csv"
combined.to_csv(combined_file, index=False)

# 5. Print a final summary
print("\n" + "=" * 50)
print("DOWNLOAD SUMMARY")
print("=" * 50)
print(f"Tickers requested: {', '.join(TICKERS)}")
print(f"Tickers downloaded: {combined['Ticker'].nunique()}")
print(f"Total rows: {len(combined):,}")
print(f"Unique dates: {combined['Date'].nunique():,}")
print(f"Data downloaded through: {combined['Date'].max().date()}")
print(f"Combined file: {combined_file}")
print("\nRows per ticker:")
print(combined.groupby("Ticker").size())
print("\nFirst five rows:")
print(combined.head())


DOWNLOAD SUMMARY
Tickers requested: AAPL, MSFT, NVDA
Tickers downloaded: 3
Total rows: 3,762
Unique dates: 1,254
Data downloaded through: 2026-10-21
Combined file: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/data/raw/all_stocks_historical.csv

Rows per ticker:
Ticker
AAPL    1254
MSFT    1254
NVDA    1254
dtype: int64

First five rows:
           Date        Open        High         Low       Close     Volume  \
0    2021-10-22  146.003295  146.481217  144.979152  145.027924   58883400   
1254 2021-10-22  297.899311  298.561525  295.404015  296.709259   17449300   
2508 2021-10-22   22.722916   23.028569   22.462065   22.626341  249384000   
1    2021-10-25  145.018195  145.691204  143.984304  144.979187   50720600   
1255 2021-10-25  296.901149  296.939546  294.117946  295.720703   17554500   

     Ticker  
0      AAPL  
1254   MSFT  
2508   NVDA  
1      AAPL  
1255   MSFT  
